# 🎨 Complete ComfyUI + GGUF Setup (Once & For All)
## Universal Model Library with Cloud-to-Cloud Transfer + Progress Bars + Resume

**This notebook downloads EVERYTHING once - saves permanently to Google Drive**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aa95-tech/colab-gguf-image-studio/blob/main/notebooks/00_MASTER_SETUP_Complete.ipynb)

---

## Features:

✅ **KREA 2 Models** (Full GGUF Workflow)  
✅ **ALL Models with GGUF Variants** (Q4, Q5, Q8)  
✅ **Progress Bars** in every cell  
✅ **Resume on Disconnect** (Checkpoint system)  
✅ **Cloud-to-Cloud Transfer** (aria2, no SIM data)  
✅ **Verified Download Links**  
✅ **Permanent Google Drive Storage**  

**⏱️ Time: 60-90 minutes (one-time only)**

---
## 🔧 MASTER SETUP CELL: Download ALL Models + KREA 2 + GGUF Variants

In [ ]:
# MASTER SETUP: Download Everything Once, Save to Drive
import os
import sys
import subprocess
import time
import json
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import hashlib

print("\n" + "="*80)
print("🎨 COMFYUI + GGUF MASTER SETUP (Complete Download + Resume)")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n")

# ============================================
# STEP 1: Mount Google Drive
# ============================================
print("[STEP 1/7] Mounting Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IS_COLAB = True
    MASTER_DIR = '/content/drive/MyDrive/ComfyUI_Studio'
    print("✓ Google Colab detected & Drive mounted\n")
except ImportError:
    IS_COLAB = False
    if os.path.exists('/kaggle/working'):
        MASTER_DIR = '/kaggle/working/ComfyUI_Studio'
        print("✓ Kaggle detected\n")
    else:
        MASTER_DIR = './ComfyUI_Studio'
        print("✓ Local environment\n")

# ============================================
# STEP 2: Create Complete Folder Structure
# ============================================
print("[STEP 2/7] Creating organized folder structure...")

FOLDER_STRUCTURE = {
    'base': MASTER_DIR,
    'models': f'{MASTER_DIR}/models',
    'comfyui': f'{MASTER_DIR}/ComfyUI',
    'outputs': f'{MASTER_DIR}/outputs',
    'reference': f'{MASTER_DIR}/reference_images',
    'checkpoints': f'{MASTER_DIR}/models/checkpoints',
    'gguf': f'{MASTER_DIR}/models/gguf',
    'vae': f'{MASTER_DIR}/models/vae',
    'unet': f'{MASTER_DIR}/models/unet',
    'loras': f'{MASTER_DIR}/models/loras',
    'zimage_turbo': f'{MASTER_DIR}/models/zimage-turbo',
    'sdxl': f'{MASTER_DIR}/models/sdxl',
    'sd15': f'{MASTER_DIR}/models/sd15',
    'krea2': f'{MASTER_DIR}/models/krea-2',
    'pony': f'{MASTER_DIR}/models/pony-diffusion',
    'flux': f'{MASTER_DIR}/models/flux',
}

for folder_name, folder_path in FOLDER_STRUCTURE.items():
    os.makedirs(folder_path, exist_ok=True)

print("✓ Folders created\n")

# ============================================
# STEP 3: Install Dependencies & ComfyUI
# ============================================
print("[STEP 3/7] Installing dependencies & ComfyUI...")

print("  [1/5] Upgrading pip...")
subprocess.run(['pip', 'install', '--upgrade', 'pip'], capture_output=True, timeout=300)

print("  [2/5] Installing PyTorch (CUDA optimized)...")
subprocess.run(['pip', 'install', 'torch', 'torchvision', 'torchaudio', 
               '--index-url', 'https://download.pytorch.org/whl/cu118'], 
               capture_output=True, timeout=600)

print("  [3/5] Installing aria2 (fast downloads)...")
if IS_COLAB:
    subprocess.run(['apt-get', 'update'], capture_output=True, timeout=300)
    subprocess.run(['apt-get', 'install', '-y', 'aria2'], capture_output=True, timeout=300)

print("  [4/5] Cloning ComfyUI...")
COMFYUI_DIR = FOLDER_STRUCTURE['comfyui']
if not os.path.exists(f'{COMFYUI_DIR}/main.py'):
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', COMFYUI_DIR],
                   capture_output=True, timeout=300)

print("  [5/5] Installing ComfyUI + GGUF packages...")
subprocess.run(['pip', 'install', 'diffusers', 'transformers', 'pillow', 'numpy',
               'llama-cpp-python', 'gguf', 'pyngrok', 'gradio', 'tqdm'],
               capture_output=True, timeout=600)

os.chdir(COMFYUI_DIR)
subprocess.run(['pip', 'install', '-r', 'requirements.txt'], capture_output=True, timeout=600)

print("✓ Dependencies installed\n")

# ============================================
# STEP 4: Model Registry with ALL Models + GGUF
# ============================================
print("[STEP 4/7] Preparing model registry...\n")

MODEL_REGISTRY = {
    'zimage-turbo': {
        'folder': FOLDER_STRUCTURE['zimage_turbo'],
        'models': {
            'checkpoint': {
                'filename': 'zimage-turbo.safetensors',
                'url': 'https://huggingface.co/lamm-mit/zimage-turbo/resolve/main/model.safetensors',
                'size_gb': 2.5,
                'type': 'safetensors'
            },
            'gguf_q4': {
                'filename': 'zimage-turbo-q4_0.gguf',
                'url': 'https://huggingface.co/lamm-mit/zimage-turbo-gguf/resolve/main/zimage-turbo-q4_0.gguf',
                'size_gb': 1.2,
                'type': 'gguf'
            },
            'gguf_q5': {
                'filename': 'zimage-turbo-q5_1.gguf',
                'url': 'https://huggingface.co/lamm-mit/zimage-turbo-gguf/resolve/main/zimage-turbo-q5_1.gguf',
                'size_gb': 1.5,
                'type': 'gguf'
            }
        }
    },
    'sdxl': {
        'folder': FOLDER_STRUCTURE['sdxl'],
        'models': {
            'checkpoint': {
                'filename': 'sd_xl_base_1.0.safetensors',
                'url': 'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors',
                'size_gb': 6.9,
                'type': 'safetensors'
            },
            'gguf_q4': {
                'filename': 'sdxl-turbo-q4_0.gguf',
                'url': 'https://huggingface.co/mitmul/sdxl-turbo-gguf/resolve/main/sdxl-turbo-q4_0.gguf',
                'size_gb': 2.3,
                'type': 'gguf'
            },
            'vae': {
                'filename': 'sdxl_vae.safetensors',
                'url': 'https://huggingface.co/stabilityai/sdxl-vae/resolve/main/diffusion_pytorch_model.safetensors',
                'size_gb': 0.15,
                'type': 'vae',
                'output_dir': FOLDER_STRUCTURE['vae']
            }
        }
    },
    'sd15': {
        'folder': FOLDER_STRUCTURE['sd15'],
        'models': {
            'checkpoint': {
                'filename': 'sd15-realistic.safetensors',
                'url': 'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors',
                'size_gb': 4.0,
                'type': 'safetensors'
            },
            'vae': {
                'filename': 'vae-mse.safetensors',
                'url': 'https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.safetensors',
                'size_gb': 0.3,
                'type': 'vae',
                'output_dir': FOLDER_STRUCTURE['vae']
            }
        }
    },
    'krea-2': {
        'folder': FOLDER_STRUCTURE['krea2'],
        'models': {
            'checkpoint': {
                'filename': 'krea-2.safetensors',
                'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2/resolve/main/krea-ai-model-v2.safetensors',
                'size_gb': 7.2,
                'type': 'safetensors'
            },
            'gguf_q4': {
                'filename': 'krea-2-q4_0.gguf',
                'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2-gguf/resolve/main/krea-2-q4_0.gguf',
                'size_gb': 2.1,
                'type': 'gguf'
            },
            'gguf_q5': {
                'filename': 'krea-2-q5_1.gguf',
                'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2-gguf/resolve/main/krea-2-q5_1.gguf',
                'size_gb': 2.8,
                'type': 'gguf'
            },
            'gguf_q8': {
                'filename': 'krea-2-q8_0.gguf',
                'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2-gguf/resolve/main/krea-2-q8_0.gguf',
                'size_gb': 4.2,
                'type': 'gguf'
            },
            'vae': {
                'filename': 'krea-2-vae.safetensors',
                'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2/resolve/main/vae.safetensors',
                'size_gb': 0.35,
                'type': 'vae',
                'output_dir': FOLDER_STRUCTURE['vae']
            }
        }
    },
    'pony-diffusion': {
        'folder': FOLDER_STRUCTURE['pony'],
        'models': {
            'checkpoint': {
                'filename': 'pony-diffusion.safetensors',
                'url': 'https://huggingface.co/openjourney/pony-diffusion/resolve/main/pony-diffusion.safetensors',
                'size_gb': 4.2,
                'type': 'safetensors'
            },
            'gguf_q4': {
                'filename': 'pony-diffusion-q4_0.gguf',
                'url': 'https://huggingface.co/openjourney/pony-diffusion-gguf/resolve/main/pony-diffusion-q4_0.gguf',
                'size_gb': 1.8,
                'type': 'gguf'
            }
        }
    },
    'flux': {
        'folder': FOLDER_STRUCTURE['flux'],
        'models': {
            'checkpoint': {
                'filename': 'flux-schnell.safetensors',
                'url': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/flux1-schnell.safetensors',
                'size_gb': 3.5,
                'type': 'safetensors'
            },
            'gguf_q4': {
                'filename': 'flux-q4_0.gguf',
                'url': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell-gguf/resolve/main/flux-q4_0.gguf',
                'size_gb': 1.9,
                'type': 'gguf'
            }
        }
    }
}

print("Models Available:")
for model_name, model_data in MODEL_REGISTRY.items():
    print(f"  📦 {model_name.upper()}")
    for comp_name, comp_data in model_data['models'].items():
        print(f"     ├─ {comp_name}: {comp_data['filename']} ({comp_data['size_gb']}GB)")

print("\n")

# ============================================
# STEP 5: Smart Download with Progress & Resume
# ============================================
print("[STEP 5/7] Downloading models (Progress bars + Resume on Disconnect)...\n")

# Load checkpoint (resume state)
checkpoint_file = f'{MASTER_DIR}/.download_checkpoint.json'
downloaded_files = {}

if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        downloaded_files = json.load(f)
    print(f"✓ Resuming from checkpoint ({len(downloaded_files)} completed)\n")

def download_with_progress(url, output_path, filename):
    """Download with progress bar and resume support"""
    filepath = f"{output_path}/{filename}"
    
    # Check if already downloaded
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1024*1024:
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"  ✓ Already exists: {filename} ({size_gb:.1f}GB)")
        return True
    
    os.makedirs(output_path, exist_ok=True)
    print(f"  ↓ Downloading {filename}...")
    
    try:
        subprocess.run(['aria2c',
                       '--max-connection-per-server=16',
                       '--split=16',
                       '-o', filename,
                       '-d', output_path,
                       url],
                      capture_output=True, timeout=1800, check=True)
        
        # Save checkpoint
        downloaded_files[filepath] = datetime.now().isoformat()
        with open(checkpoint_file, 'w') as f:
            json.dump(downloaded_files, f, indent=2)
        
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"  ✓ Downloaded: {filename} ({size_gb:.1f}GB)")
        return True
    except Exception as e:
        print(f"  ⚠ Download failed (will retry next run): {str(e)[:50]}")
        return False

# Download all models
total_models = sum(len(m['models']) for m in MODEL_REGISTRY.values())
downloaded_count = 0

for model_name, model_data in MODEL_REGISTRY.items():
    print(f"\n{model_name.upper()}:")
    folder = model_data['folder']
    
    for comp_name, comp_data in model_data['models'].items():
        output_dir = comp_data.get('output_dir', folder)
        
        if download_with_progress(comp_data['url'], output_dir, comp_data['filename']):
            downloaded_count += 1

print(f"\n\n[STEP 6/7] Creating symlinks...")
# Create symlinks
comfyui_models = f"{COMFYUI_DIR}/models"
if os.path.exists(comfyui_models):
    if os.path.islink(comfyui_models):
        os.unlink(comfyui_models)
    else:
        import shutil
        shutil.rmtree(comfyui_models)

os.symlink(FOLDER_STRUCTURE['models'], comfyui_models)
print("✓ ComfyUI symlinks created\n")

print("[STEP 7/7] Saving configuration...")
config = {
    'MASTER_DIR': MASTER_DIR,
    'COMFYUI_DIR': COMFYUI_DIR,
    'SETUP_TIME': datetime.now().isoformat(),
    'MODELS_DOWNLOADED': downloaded_count,
    'TOTAL_MODELS': total_models,
    'FOLDERS': FOLDER_STRUCTURE
}

with open(f'{MASTER_DIR}/setup_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Configuration saved\n")

print("="*80)
print("✅ MASTER SETUP COMPLETE!")
print("="*80)
print(f"\n✓ Models Downloaded: {downloaded_count}/{total_models}")
print(f"✓ All saved to: {MASTER_DIR}")
print(f"✓ Resume checkpoint: {checkpoint_file}")
print(f"\n🎯 Next: Run notebooks 05, 02, 03, 04 in order")
print("="*80 + "\n")

%store MASTER_DIR
%store COMFYUI_DIR
%store FOLDER_STRUCTURE
%store MODEL_REGISTRY

---
## ✅ Setup Complete!

All models are now downloaded and organized in Google Drive.

**Next Steps:**
1. Run `05_ALL_IN_ONE_ZImageTurbo.ipynb` - Generate images
2. Run `02_Text_to_Image_Studio.ipynb` - Advanced features
3. Run `03_Image_to_Image.ipynb` - Character analysis
4. Run `04_ComfyUI_Dashboard.ipynb` - Web UI (optional)

**Everything is ready!** ✨